In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
data = [("101", "2023-12-01", 100), ("101", "2023-12-02", 150), ("102", "2023-12-01", 200), ("102", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

product_id,date,sales
101,2023-12-01,100
101,2023-12-02,150
102,2023-12-01,200
102,2023-12-02,250


In [0]:
df1 = df.withColumn("date", to_date("date")).orderBy(col("product_id").asc(), col("date").desc()).dropDuplicates(["product_id"]).display()


product_id,date,sales
101,2023-12-02,150
102,2023-12-02,250


###5 You need to calculate the total number of actions performed by users in a system. How would you calculate the top 5 most active users based on this information?

In [0]:
data = [("user1", 5), ("user2", 8), ("user3", 2), ("user4", 10), ("user2", 3)]
columns = ["user_id", "actions"]

df = spark.createDataFrame(data, columns)
df.display()

user_id,actions
user1,5
user2,8
user3,2
user4,10
user2,3


In [0]:
#df.groupBy("user_id").agg(sum("actions").alias("noOfActions")).orderBy(col("noOfActions").disc()).display()
df.groupBy("user_id").agg(sum("actions").alias("noOfActions")).orderBy("noOfActions",ascending=False).display()

user_id,noOfActions
user2,11
user4,10
user1,5
user3,2


### 6. While processing sales transaction data, you need to identify the most recent transaction for each customer. How would you approach this task?

In [0]:
data = [
    ("cust1", "2023-12-01", 100),
    ("cust2", "2023-12-02", 150),
    ("cust1", "2023-12-03", 200),
    ("cust2", "2023-12-04", 250)
]
columns = ["customer_id", "transaction_date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

customer_id,transaction_date,sales
cust1,2023-12-01,100
cust2,2023-12-02,150
cust1,2023-12-03,200
cust2,2023-12-04,250


In [0]:
df.withColumn("transaction_date",to_date("transaction_date")).withColumn("rank",dense_rank().over(Window.partitionBy("customer_id").orderBy(col("customer_id").asc(),col("transaction_date").desc()))).where ("rank=1").display()

customer_id,transaction_date,sales,rank
cust1,2023-12-03,200,1
cust2,2023-12-04,250,1


#7. You need to identify customers who haven't made any purchases in the last 30 days. How would you filter such customers?

In [0]:
data = [
    ("cust1", "2025-12-01"),
    ("cust2", "2024-11-20"),
    ("cust3", "2024-11-25")
]
columns = ["customer_id", "last_purchase_date"]

df = spark.createDataFrame(data, columns)

df.display()

customer_id,last_purchase_date
cust1,2025-12-01
cust2,2024-11-20
cust3,2024-11-25


In [0]:
df.withColumn("last_purchase_date",to_date("last_purchase_date")).filter(col("last_purchase_date") < current_date()-30).display()

customer_id,last_purchase_date
cust1,2025-12-01
cust2,2024-11-20
cust3,2024-11-25


##8. While analyzing customer reviews, you need to identify the most frequently used words in the feedback. How would you implement this?

In [0]:
data = [
    ("customer1", "The product is great"),
    ("customer2", "Great product, fast delivery"),
    ("customer3", "Not bad, could be better")
]
columns = ["customer_id", "feedback"]

df = spark.createDataFrame(data, columns)

df.display()

customer_id,feedback
customer1,The product is great
customer2,"Great product, fast delivery"
customer3,"Not bad, could be better"


In [0]:
df=df.withColumn("feedback",lower("feedback")).withColumn("feedback",explode(split("feedback"," "))).display()
#df_grp=df.groupBy("feedback").agg(count("feedback").alias("wrdcnt")).orderBy(col("wrdcnt").desc()).display()

customer_id,feedback
customer1,the
customer1,product
customer1,is
customer1,great
customer2,great
customer2,"product,"
customer2,fast
customer2,delivery
customer3,not
customer3,"bad,"


##9. You need to calculate the cumulative sum of sales over time for each product. How would you approach this?

In [0]:
data = [
    ("product1", "2023-12-01", 100),
    ("product2", "2023-12-02", 200),
    ("product1", "2023-12-03", 150),
    ("product2", "2023-12-04", 250)
]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

product_id,date,sales
product1,2023-12-01,100
product2,2023-12-02,200
product1,2023-12-03,150
product2,2023-12-04,250


In [0]:
df=df.withColumn("date",to_date("date"))
df=df.withColumn("cumusum",sum("sales").over(Window.partitionBy("product_id").orderBy("date"))).display()

product_id,date,sales,cumusum
product1,2023-12-01,100,100
product1,2023-12-03,150,250
product2,2023-12-02,200,200
product2,2023-12-04,250,450


##10. While preparing a data pipeline, you notice some duplicate rows in a dataset. How would you remove the duplicates without affecting the original order?

In [0]:
data = [("John", 25), ("Jane", 30), ("John", 25), ("Alice", 22)]
columns = ["name", "age"]
df = spark.createDataFrame(data, columns)
df.display()

name,age
John,25
Jane,30
John,25
Alice,22


In [0]:
df=df.withColumn("rnk",row_number().over(Window.partitionBy("name").orderBy("age"))).filter("rnk=1").display()

name,age,rnk
Alice,22,1
Jane,30,1
John,25,1


##11. You are working with user activity data and need to calculate the average session duration per user. How would you implement this?

In [0]:
data = [("user1", "2023-12-01", 50), ("user1", "2023-12-02", 60),
        ("user2", "2023-12-01", 45), ("user2", "2023-12-03", 75)]
columns = ["user_id", "session_date", "duration"]
df = spark.createDataFrame(data, columns)

df.display()

user_id,session_date,duration
user1,2023-12-01,50
user1,2023-12-02,60
user2,2023-12-01,45
user2,2023-12-03,75


In [0]:
df=df.groupBy("user_id").agg(avg("duration").alias("avg_duration")).display()

user_id,avg_duration
user1,55.0
user2,60.0


##12. While analyzing sales data, you need to find the product with the highest sales for each month. How would you accomplish this?

In [0]:
data = [("product1", "2023-12-01", 100), ("product2", "2023-12-01", 150),
        ("product1", "2023-12-02", 200), ("product2", "2023-12-02", 250)]
columns = ["product_id", "date", "sales"]
df = spark.createDataFrame(data, columns)
df.display()

product_id,date,sales
product1,2023-12-01,100
product2,2023-12-01,150
product1,2023-12-02,200
product2,2023-12-02,250


In [0]:
df=df.withColumn("date",to_date(col("date"))).withColumn("month",month(col("date")))
df1=df.groupBy("product_id","month").agg(sum("sales").alias("mnthsal"))
df2=df1.withColumn("rnk",row_number().over(Window.partitionBy("product_id").orderBy(col("mnthsal").desc()))).display()

product_id,month,mnthsal,rnk
product1,12,300,1
product2,12,400,1


##13. You are working with a large Delta table that is frequently updated by multiple users. The data is stored in partitions, and sometimes updates can cause inconsistent reads due to concurrent transactions. How would you ensure ACID compliance and avoid data corruption in PySpark?

In [0]:
_delta_log and merge command

##14. You need to process a large dataset stored in PARQUET format and ensure that all columns have the right schema (Almost). How would you do this?

In [0]:
df = spark.read.format('parquet')
     .option('inferSchema',True)
     .load('path')

##15. You are reading a CSV file and need to handle corrupt records gracefully by skipping them. How would you configure this in PySpark?

In [0]:
df = spark.read.format("csv") \
    .option("mode", "DROPMALFORMED") \
    .load("staging_location")

#22. You have a dataset containing the names of employees and their departments. You need to find the department with the most employees.

In [0]:
data = [("Alice", "HR"), ("Bob", "Finance"), ("Charlie", "HR"), ("David", "Engineering"), ("Eve", "Finance")]
columns = ["employee_name", "department"]

df = spark.createDataFrame(data, columns)
df.display()

employee_name,department
Alice,HR
Bob,Finance
Charlie,HR
David,Engineering
Eve,Finance


In [0]:
df=df.groupBy("department").agg(count("*").alias("noOfEmp"))\
.withColumn("rnk",dense_rank().over(Window.orderBy(col("noOfEmp").desc())))\
.filter("rnk=1").display()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


department,noOfEmp,rnk
HR,2,1
Finance,2,1


##23. While processing sales data, you need to classify each transaction as either 'High' or 'Low' based on its amount. How would you achieve this using a when condition



In [0]:
data = [("product1", 100), ("product2", 300), ("product3", 50)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,sales
product1,100
product2,300
product3,50


In [0]:
df=df.withColumn("clasify", when(df.sales>200,"hight").otherwise("low") ).display()

product_id,sales,clasify
product1,100,low
product2,300,hight
product3,50,low


##24. While analyzing a large dataset, you need to create a new column that holds a timestamp of when the record was processed. How would you implement this and what can be the best USE CASE?

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,sales
product1,100
product2,200
product3,300


In [0]:
df=df.withColumn("processedtime",current_timestamp()).display()

product_id,sales,processedtime
product1,100,2026-04-18T05:14:27.429Z
product2,200,2026-04-18T05:14:27.429Z
product3,300,2026-04-18T05:14:27.429Z


##25. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it. How would you achieve this?

In [0]:
data = [("product1", 100), ("product2", 200), ("product3", 300)]
columns = ["product_id", "sales"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,sales
product1,100
product2,200
product3,300


In [0]:
df.createOrReplaceTempView("df_sql")

In [0]:
spark.sql("select * from df_sql").display()


product_id,sales
product1,100
product2,200
product3,300


In [0]:
%sql
select * from df_sql

product_id,sales
product1,100
product2,200
product3,300


##26. You need to register this PySpark DataFrame as a temporary SQL object and run a query on it (FROM DIFFERENT NOTEBOOKS AS WELL)?

In [0]:
df.createOrReplaceGlobalTempView("df_sql_global")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7386627243737046>, line 1
----> 1 df.createOrReplaceGlobalTempView("df_sql_global")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2129, in DataFrame.createOrReplaceGlobalTempView(self, name)
   2125 def createOrReplaceGlobalTempView(self, name: str) -> None:
   2126     command = plan.CreateView(
   2127         child=self._plan, name=name, is_global=True, replace=True
   2128     ).command(session=self._session.client)
-> 2129     _, _, ei = self._session.client.execute_command(command, self._plan.observations)
   2130     self._execution_info = ei

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/client/core.py:1538, in SparkConnectClient.execute_command(self, command, observations, extra_request_metadata)
   1536     req.user_context.user_id = self._user_i

In [0]:
%sql
SELECT * FROM global_temp.globalview;

##27. You need to query data from a PySpark DataFrame using SQL, but the data includes a nested structure. How would you flatten the data for easier querying?

In [0]:
data = [("product1", {"price": 100, "quantity": 2}),
        ("product2", {"price": 200, "quantity": 3})]
columns = ["product_id", "product_info"]

df = spark.createDataFrame(data, columns)
df.display()

product_id,product_info
product1,"Map(price -> 100, quantity -> 2)"
product2,"Map(price -> 200, quantity -> 3)"


In [0]:
df.select("product_id", "product_info.price", "product_info.quantity").display()

product_id,price,quantity
product1,100,2
product2,200,3


##29. While reading data from Parquet, you need to optimize performance by partitioning the data based on a column. How would you implement this?

In [0]:
df.write.format("parquet").mode("append").partitionBy("category").save("location")

##30. You are working with a large dataset in Parquet format and need to ensure that the data is written in an optimized manner with proper compression. How would you accomplish this?

In [0]:
df.write.format("parquet").option("compression","snappy")

##31. Your company uses a large-scale data pipeline that reads from Delta tables and processes data using complex aggregations. However, performance is becoming an issue due to the growing dataset size. How would you optimize the performance of the pipeline?

In [0]:
1  %sql
2
3  OPTIMIZE tabledelta ZORDER BY ('order_date')

##What are broadcast variables, and why are they used?

##What is the difference between df.show() and df.collect()

## show() --> It returns few records
## collect() --> It returns all the records to the driver
#show() → displays a few rows (default 20) for quick inspection, doesn't return data
#collect() → returns all rows to driver memory as a Python list (risky for large datasets)

##What are the advantages of Delta Lake over traditional file formats?

##What happens when a PySpark job runs out of memory? 

#What is spill in Spark, and why does it happen?

##43. You are processing sales data. Group by product categories and create a list of all product names in each category.

In [0]:
data = [("Electronics", "Laptop"), ("Electronics", "Smartphone"), ("Furniture", "Chair"), ("Furniture", "Table")]
columns = ["category", "product"]
df = spark.createDataFrame(data, columns)
df.display()

category,product
Electronics,Laptop
Electronics,Smartphone
Furniture,Chair
Furniture,Table


In [0]:
df=df.groupBy("category").agg(collect_list("product").alias("product")).display()

category,product
Electronics,"List(Laptop, Smartphone)"
Furniture,"List(Chair, Table)"


##44. You are analyzing orders. Group by customer IDs and list all unique product IDs each customer purchased.

In [0]:
data = [[101, "P001"], [101, "P002"], [102, "P001"], [101, "P001"]]
columns = ["customer_id", "product_id"]
df = spark.createDataFrame(data, columns)
df.display()

customer_id,product_id
101,P001
101,P002
102,P001
101,P001


In [0]:
df=df.groupBy("customer_id").agg(collect_set("product_id").alias("product_id")).display()

customer_id,product_id
101,"List(P001, P002)"
102,List(P001)


##45. For customer records, combine first and last names only if the email address exists.

In [0]:
data = [("John", "Doe", "john.doe@example.com"), ("Jane", "Smith", None)]
columns = ["first_name", "last_name", "email"]
df = spark.createDataFrame(data, columns)
df.display()

first_name,last_name,email
John,Doe,john.doe@example.com
Jane,Smith,null


In [0]:
df=df.withColumn("fullname", when(df.email.isNotNull(),concat(df.first_name, lit(" "), df.last_name))).display()

first_name,last_name,email,fullname
John,Doe,john.doe@example.com,John Doe
Jane,Smith,null,null


In [0]:
df = df.withColumn("fullname",when(col('email').isNotNull(),concat_ws("-",col('first_name'),col('last_name'))).otherwise(None)

##46. You have a DataFrame containing customer IDs and a list of their purchased product IDs. Calculate the number of products each customer has purchased.

In [0]:
data = [
    (1, ["prod1", "prod2", "prod3"]),
    (2, ["prod4"]),
    (3, ["prod5", "prod6"]),
]

myschema = "customer_id INT ,product_ids array<STRING>"

df = spark.createDataFrame(data, myschema)
df.display()

customer_id,product_ids
1,"List(prod1, prod2, prod3)"
2,List(prod4)
3,"List(prod5, prod6)"


In [0]:
df=df.withColumn("size",size(col("product_ids"))).display()

customer_id,product_ids,size
1,"List(prod1, prod2, prod3)",3
2,List(prod4),1
3,"List(prod5, prod6)",2


##47. You have employee IDs of varying lengths. Ensure all IDs are 6 characters long by padding with leading zeroes.

In [0]:
data = [
    ("1",),
    ("123",),
    ("4567",),
]
schema = ["employee_id"]

df = spark.createDataFrame(data, schema)
df.display()

employee_id
1
123
4567


In [0]:
df=df.withColumn("employee_id",lpad(col("employee_id"),6,"0")).display()

employee_id
000001
000123
004567


##48. You need to validate phone numbers by checking if they start with "91"

In [0]:
data = [
    ("911234567890",),
    ("811234567890",),
    ("912345678901",),
]
schema = ["phone_number"]

df = spark.createDataFrame(data, schema)
df.display()

phone_number
911234567890
811234567890
912345678901


In [0]:
#df = df.where(col("phone_number").startswith("91")).display()
df=df.filter(substring(col("phone_number"),1,2)=="91").display()


phone_number
911234567890
912345678901


##49. You have a dataset with courses taken by students. Calculate the average number of courses per student.

In [0]:
data = [
    (1, ["Math", "Science"]),
    (2, ["History"]),
    (3, ["Art", "PE", "Biology"]),
]
schema = ["student_id", "courses"]

df = spark.createDataFrame(data, schema)
df.display()

student_id,courses
1,"List(Math, Science)"
2,List(History)
3,"List(Art, PE, Biology)"


In [0]:
df=df.withColumn("size",size(col("courses")).alias("size")).agg(avg("size")).display()


avg(size)
2.0


#50. You have a dataset with primary and secondary contact numbers. Use the primary number if available; otherwise, use the secondary number.

In [0]:
data = [
    (None, "1234567890"),
    ("9876543210", None),
    ("7894561230", "4567891230"),
]
schema = ["primary_contact", "secondary_contact"]

df = spark.createDataFrame(data, schema)
df.display()

primary_contact,secondary_contact
null,1234567890
9876543210,null
7894561230,4567891230


In [0]:
df=df.withColumn("contact",coalesce(col("primary_contact"),col("secondary_contact"))).display()

primary_contact,secondary_contact,contact
null,1234567890,1234567890
9876543210,null,9876543210
7894561230,4567891230,7894561230


#51. You are categorizing product codes based on their lengths. If the length is 5, label it as "Standard"; otherwise, label it as "Custom".

In [0]:
data = [
    ("prod1",),
    ("prd23a",),
    ("pr987c",),
]
schema = ["product_code"]

df = spark.createDataFrame(data, schema)
df.display()

product_code
prod1
prd23a
pr987c


In [0]:
df=df.withColumn("label",when(length(col("product_code"))==5,"standard").otherwise("custom")).display()

product_code,label
prod1,standard
prd23a,custom
pr987c,custom


#You are given a dataset with the following columns:
#customer_id, order_date, and revenue.

#Remove records where revenue <= 0 or customer_id is NULL
#For each customer, consider only the latest order
#Calculate the day-wise total revenue based on the filtered data

In [0]:
data = [
    (1, "2024-01-01", 100),
    (1, "2024-01-05", 200),   # latest for customer 1
    (2, "2024-01-02", -50),   # invalid (<=0)
    (2, "2024-01-03", 150),
    (3, "2024-01-04", 0),     # invalid (<=0)
    (None, "2024-01-05", 300),# invalid (customer_id null)
    (4, "2024-01-06", 250)
]

df = spark.createDataFrame(data, ["customer_id", "order_date", "revenue"])
display(df)

customer_id,order_date,revenue
1,2024-01-01,100
1,2024-01-05,200
2,2024-01-02,-50
2,2024-01-03,150
3,2024-01-04,0
null,2024-01-05,300
4,2024-01-06,250


In [0]:
from pyspark.sql.functions import col, to_date, sum, row_number
from pyspark.sql.window import Window

# Step 1: Filter invalid records
df1 = df.filter(
    (col("revenue") > 0) & 
    (col("customer_id").isNotNull())
)

# Step 2: Convert to date (if needed)
df1 = df1.withColumn("order_date", to_date(col("order_date")))

# Step 3: Keep latest record per customer
window = Window.partitionBy("customer_id").orderBy(col("order_date").desc())

df2 = df1.withColumn("rn", row_number().over(window)) \
         .filter(col("rn") == 1) \
         .drop("rn")

# Step 4: Day-wise revenue
result = df2.groupBy("order_date") \
            .agg(sum("revenue").alias("daily_revenue"))

display(result)

order_date,daily_revenue
2024-01-05,200
2024-01-03,150
2024-01-06,250


In [0]:
WITH filtered_data AS (
    -- Step 1: Filter invalid records
    SELECT *
    FROM table_name
    WHERE revenue > 0
      AND customer_id IS NOT NULL
),

converted_data AS (
    -- Step 2: Convert to date
    SELECT 
        customer_id,
        TO_DATE(order_date) AS order_date,
        revenue
    FROM filtered_data
),

latest_records AS (
    -- Step 3: Keep latest record per customer
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_id 
               ORDER BY order_date DESC
           ) AS rn
    FROM converted_data
)

-- Step 4: Day-wise revenue
SELECT 
    order_date,
    SUM(revenue) AS daily_revenue
FROM latest_records
WHERE rn = 1
GROUP BY order_date;

#You are given a dataset of transactions with the following columns:
#transaction_id/account_id/amount/transaction_time/location
#Some transactions may be duplicates if they occur within a short time interval.

In [0]:
data = [
    ("T001", "A123", 100.0, "2023-09-01 10:00:00", "NY"),
    ("T002", "A123", 100.0, "2023-09-01 10:01:30", "NY"),  # Duplicate
    ("T003", "A123", 100.0, "2023-09-01 10:05:00", "NY"),  # Not duplicate
   
    ("T004", "A124", 200.0, "2023-09-01 11:00:00", "LA"),
    ("T005", "A124", 200.0, "2023-09-01 11:01:00", "LA"),  # Duplicate
]
columns = ["transaction_id", "account_id", "amount", "c", "location"]

In [0]:
df=spark.createDataFrame(data,columns)
df.withColumn("c",to_timestamp("c")).withColumn("previou_date",lag("c").over(Window.partitionBy("account_id").orderBy("c"))).withColumn("difference",(unix_timestamp(col("c"))-unix_timestamp(col("previou_date")))/60).withColumn("status",when(col("difference") < 2,"duplicate").otherwise("not duplicate")).display()



transaction_id,account_id,amount,c,location,previou_date,difference,status
T001,A123,100.0,2023-09-01T10:00:00.000Z,NY,null,null,not duplicate
T002,A123,100.0,2023-09-01T10:01:30.000Z,NY,2023-09-01T10:00:00.000Z,1.5,duplicate
T003,A123,100.0,2023-09-01T10:05:00.000Z,NY,2023-09-01T10:01:30.000Z,3.5,not duplicate
T004,A124,200.0,2023-09-01T11:00:00.000Z,LA,null,null,not duplicate
T005,A124,200.0,2023-09-01T11:01:00.000Z,LA,2023-09-01T11:00:00.000Z,1.0,duplicate
